# Data Analysis

Answer the selected question, “Which delivery time slots cause the most late orders?”, using approval time as the time-slot anchor. Delivery outcomes count distinct orders.

**Contents**

1. [Project setup and input](#analysis-setup) — Load the cleaned item table.
2. [Order-level delivery outcomes](#analysis-orders) — Classify orders, assign approval slots and calculate lateness.
3. [Late delivery by approval slot](#analysis-slot-summary) — Compare order counts, late rates and delay duration.

Section 2 uses the table loaded in section 1.

Input dataset documentation: [Cleaning User Guide](../docs/Cleaning/User_Guide.md) · [Cleaning Technical Reference](../docs/Cleaning/Technical_Reference.md)


<a id="analysis-setup" name="analysis-setup"></a>

## 1. Project setup and input

Locate the project and load the cleaned order-item table.


In [ ]:
from pathlib import Path
import sys

from IPython.display import display


def find_project_root(start_path: Path) -> Path:
    # Resolve imports from either the repository root or notebooks directory.
    for path in [start_path, *start_path.parents]:
        if (path / "src").is_dir() and (path / "requirements.txt").is_file():
            return path
    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from src.analysis.orders import build_order_analysis_view
from src.analysis.summary import summarize_delivery_by_slot
from src.data_loader import load_data
from src.paths import PROCESSED_DIR

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed input: {PROCESSED_DIR}")


### Load the cleaned item table


In [ ]:
cleaned_items = load_data("cleaned_order_items", stage="processed")


<a id="analysis-orders" name="analysis-orders"></a>

## 2. Order-level delivery outcomes

Keep each order once in the analysis view. Orders with missing or conflicting order-level values retain all their item rows in the review view. One reviewed order may have several reasons. Approval time determines one of four six-hour slots. A delivery is late only when its calendar date is after the estimated date.


In [ ]:
orders, order_diagnostics = build_order_analysis_view(cleaned_items)
display(order_diagnostics["cohort_summary"])

if not order_diagnostics["review_reasons"].empty:
    print("Orders requiring review by field and issue")
    display(order_diagnostics["review_reasons"].groupby(["field", "issue"]).size().rename("orders").reset_index())
    print("First 20 review reasons; complete records remain in order_diagnostics['review_reasons'].")
    display(order_diagnostics["review_reasons"].head(20))
    print("Original item rows remain in order_diagnostics['review_items'].")

print("First five analyzable orders")
display(orders[[
    "order_id", "order_approved_at", "approval_slot",
    "order_estimated_delivery_date", "order_delivered_customer_date",
    "is_late", "delay_days",
]].head())


<a id="analysis-slot-summary" name="analysis-slot-summary"></a>

## 3. Late delivery by approval slot

Compare the four approval-time slots at order grain. Late rate divides late orders by all analyzable orders in the slot. Mean and median delay use late orders only.


In [ ]:
slot_summary = summarize_delivery_by_slot(orders)
display(slot_summary.style.format({
    "orders": "{:,.0f}",
    "on_time_orders": "{:,.0f}",
    "late_orders": "{:,.0f}",
    "late_rate_pct": "{:.2f}%",
    "mean_delay_days": "{:.2f}",
    "median_delay_days": "{:.0f}",
}, na_rep="—").hide(axis="index"))

assert slot_summary["orders"].sum() == len(orders)
assert slot_summary["late_orders"].sum() == orders["is_late"].sum()

if orders.empty:
    print("No analyzable orders are available for slot comparisons.")
elif not orders["is_late"].any():
    print("No late orders are available for a late-delivery ranking.")
else:
    for label, measure in [
        ("Most late orders", "late_orders"),
        ("Highest late rate", "late_rate_pct"),
        ("Longest mean delay among late orders", "mean_delay_days"),
    ]:
        maximum = slot_summary[measure].max()
        leaders = slot_summary.loc[slot_summary[measure].eq(maximum), "approval_slot"]
        print(f"{label}: {', '.join(leaders)}")


For the current cleaned dataset, 12:00–18:00 has the most late orders (2,085) and the largest order volume. The highest late rate is 00:00–06:00 (7.31%). Among late orders, 06:00–12:00 has the longest mean delay (11.18 days), while every slot has a median of seven days. These rankings describe different aspects of delivery performance.
